# Portfolio charts
Generates three images for the Upwork portfolio entry and README. Images are saved next to this notebook (PNG, 150 dpi).

In [ ]:
# ---- Cell 1: load, clean, split (same as notebook 03) ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["MonthlyCharges"] * df["tenure"])
df["churned"] = (df["Churn"] == "Yes").astype(int)

X = df.drop(columns=["customerID", "Churn", "churned"])
y = df["churned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat_cols = [c for c in X.columns if c not in num_cols]
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
model = Pipeline([("prep", preprocessor),
                  ("clf", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]

plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})


In [ ]:
# ---- Cell 2: Image 1 — the business result ----
base_rate = y_test.mean()
k = int(len(y_test) * 0.20)
top_idx = np.argsort(probs)[::-1][:k]
model_rate = y_test.iloc[top_idx].mean()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(["Random 20% of customers", "Model's top 20% riskiest"],
              [base_rate * 100, model_rate * 100], color=["#B0B0B0", "#2E86AB"])
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1,
            f"{b.get_height():.0f}%", ha="center", fontsize=16, fontweight="bold")
ax.set_ylabel("Share who actually churned (%)")
ax.set_ylim(0, 100)
ax.set_title(f"Calling the model's top 20% reaches {model_rate / base_rate:.1f}x more churners")
plt.tight_layout()
plt.savefig("01_business_result.png", dpi=150)
plt.show()


In [ ]:
# ---- Cell 3: Image 2 — churn drivers from the EDA ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

contract = df.groupby("Contract")["churned"].mean().mul(100).sort_values(ascending=False)
axes[0].bar(contract.index, contract.values, color="#2E86AB")
axes[0].set_title("Churn rate by contract type")

df["tenure_group"] = pd.cut(df["tenure"], bins=[0, 12, 24, 36, 48, 60, 72],
                            labels=["0-12", "13-24", "25-36", "37-48", "49-60", "61-72"],
                            include_lowest=True)
tenure = df.groupby("tenure_group", observed=True)["churned"].mean().mul(100)
axes[1].bar(tenure.index.astype(str), tenure.values, color="#2E86AB")
axes[1].set_title("Churn rate by tenure (months)")

for ax, s in [(axes[0], contract), (axes[1], tenure)]:
    ax.axhline(df["churned"].mean() * 100, color="gray", linestyle="--", label="Overall rate")
    ax.set_ylabel("Churn rate (%)")
    for i, v in enumerate(s.values):
        ax.text(i, v + 1, f"{v:.0f}%", ha="center")
axes[0].legend()
plt.tight_layout()
plt.savefig("02_churn_drivers.png", dpi=150)
plt.show()


In [ ]:
# ---- Cell 4: Image 3 — ROC curve vs random ----
fpr, tpr, _ = roc_curve(y_test, probs)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, color="#2E86AB", lw=2,
        label=f"Logistic regression (AUC = {roc_auc_score(y_test, probs):.2f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random guess (AUC = 0.50)")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curve on 1,409 unseen customers")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("03_roc_curve.png", dpi=150)
plt.show()
